# 20 · 大考：每个部件值多少分

## 这一章在整个项目里的位置

```
00 总览 → 01 存档旧项目 ✅ → 02 新项目骨架 ✅ → 03 自动检查 ✅ → 04 GitHub 与 CI ✅ → 05 准备考题 ✅ → 06 评测集 ✅ → 07 判分 ✅ → 08 MLflow 评测 ✅ → 09 第一个基线 ✅ → 10 第一个工具 ✅ → 11 第二个工具 ✅ → 12 第三个工具 ✅ → 13 标准接口 MCP ✅ → 14 固定流程 ✅ → 15 第一个 agent ✅ → 16 看错题 ✅ → 17 找相似病人 ✅ → 18 疾病介绍（RAG） ✅ → 19 基因工具 ✅ → [20 大考 ✅] → 21 做成服务
```

> **这一章的代码在合并请求 #22 里。** 大考和判断实验一共花了约 ¥45（其中一次超支，第 7 步讲），全部记在 MLflow 里。下面每一格都只读记录、在本机计算，**运行这本笔记不花钱**。

## 背景：工具做完了，现在要回答三个问题

第 16 章定了计划：错题分析告诉我们缺什么，第 17–19 章把工具一个个做出来、单独考过。现在把它们放进同一个 agent，在 dev-200 上正式考，回答三个问题：

1. **agent 到底有没有比固定流程强？** 第 15 章是 39.0% 比 35.5%，逐题比 p = 0.19，没证明
2. **每个新部件值多少分？** 不加分的删掉
3. **剩下的错误错在哪？** 下一步往哪改

## 这一章做的 8 件事

| # | 做什么 | 用的技术 | 回答什么 |
|---|---|---|---|
| 1 | 设计大考 | 消融实验：全部工具一次，再每次拿掉一个 | 每个部件值多少分 |
| 2 | 定稿 agent 和固定流程逐题比 | 符号检验 | agent 是不是真的更强 |
| 3 | 每个工具值多少分 | 消融结果 + 每道题的 token 用量 | 哪个留、哪个删 |
| 4 | 错题分析：瓶颈挪到了哪 | 追踪记录 | 下一步改什么 |
| 5 | 重放判断：便宜地试三种判断方法 | 从记录里取证据，每道题只调一次模型 | 判断能不能改进 |
| 6 | 基因考法 | 和不用模型的基因排名比 | 有基因时 agent 还有没有用 |
| 7 | 两次意外 | 接口故障检查；先小考的规矩 | 分数和花费都要靠得住 |
| 8 | 定稿前：还有没有能提高的地方 | 用已有的记录重放，不花钱 | 留出集只能考一次，考前把能试的都试了 |

---

## 准备

运行下面这一格。它定义一个叫 `run` 的小工具，用来执行命令并显示结果。**不用看代码。**

> 运行 `uv` 的命令时，有时会先出现几行 `Building` / `Installed`：那是 uv 发现代码有变化，顺手更新了一下环境，可以忽略。

In [ ]:
import os, subprocess
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
os.chdir(ROOT)
os.environ.update(NO_COLOR="1", GH_PAGER="cat", GIT_PAGER="cat")
for d in ("/opt/homebrew/bin", "/usr/local/bin", str(Path.home() / ".local/bin")):  # 编辑器不是从终端打开时，也找得到 uv
    if Path(d).is_dir() and d not in os.environ["PATH"].split(os.pathsep):
        os.environ["PATH"] += os.pathsep + d
def run(cmd, check=True):
    print(f"$ {cmd}")
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(r.stdout + r.stderr)
    if check and r.returncode != 0:
        raise RuntimeError(f"failed: {cmd}")
print("项目目录:", ROOT)

---

## 第 1 步 · 设计大考

### 在干什么
在 dev-200 上考定稿 agent，再每次拿掉一个部件各考一次。

### 背景
带上全部工具考一次，只能知道"整体多少分"。想知道每个工具值多少，就要拿掉它再考一次，看分数掉多少，这叫**消融实验**（ablation）。拿掉以后分数不掉的部件，就没必要留着：它占提示词、多花 token、还可能让 agent 分心。

### 用了什么技术，为什么
| 考什么 | 回答的问题 |
|---|---|
| 定稿 agent：新开局 + 全部工具 | 定稿系统考多少 |
| 拿掉"找相似病人" | 这个工具值多少 |
| 拿掉"查疾病介绍" | 这个工具值多少 |
| 固定流程，按第 18 章修过的防泄露规则重考 | 对照组，同一套规则 |
| 基因考法：定稿 agent | 有基因时考多少 |

每次考完自动做泄露审计和接口故障检查（第 7 步）。

### 公司里什么情况怎么做
- **消融实验是证明"每个部件都有用"的标准做法**，论文和公司内部评审都要
- **对照组要在同一套规则下重考**：规则改过（第 18 章收紧了防泄露），旧分数就不能直接拿来比

### 执行

In [ ]:
from raredx.config import settings
from raredx.evals.harness import Harness

harness = Harness(settings.mlflow_tracking_uri, settings.mlflow_artifact_root)
def latest(name, split="dev-200"):
    # A run on uncommitted code is used only if it was reviewed and tagged as such.
    for r in harness.runs(split):
        if r["name"] == name and (not r["code"].endswith("-dirty")
                                  or "code_reviewed" in harness.client.get_run(r["run_id"]).data.tags):
            return r

rows = [("固定流程", "fixed-pipeline"),
        ("定稿 agent（全部工具）", "seeded-agent"),
        ("拿掉 找相似病人", "seeded-agent (without find_similar_patients)"),
        ("拿掉 查疾病介绍", "seeded-agent (without search_disease_texts)")]
runs = {}
for label, name in rows:
    runs[name] = record = latest(name)
    run_data = harness.client.get_run(record["run_id"]).data     # 不叫 run：那是"准备"里的小工具
    m = run_data.metrics
    tokens = m.get("usage/input_tokens", 0) / 200
    print(f"{label:<14} 第 1 名 {m['disease_top1/mean']:.1%}   前 10 名 {m['disease_top10/mean']:.1%}   "
          f"每道题输入 {tokens:>6,.0f} token   泄露审计 {run_data.tags.get('leak_audit')}")

### 结果怎么看

| dev-200，200 道题 | 第 1 名 | 前 10 名 |
|---|---|---|
| 固定流程 | 35.5% | 54.5% |
| **定稿 agent（全部工具）** | **41.5%** | **59.0%** |
| 拿掉"找相似病人" | 38.5% | 59.5% |
| 拿掉"查疾病介绍" | 40.5% | 58.5% |

- 四次考试都通过了泄露审计
- 拿掉"查疾病介绍"，每道题少用约 18% 的 token
- "拿掉查疾病介绍"那次，开考时我正在同一个文件夹里写第 5 步的代码，没提交，所以记录的代码版本带着 `-dirty`。我复核过：没提交的只是第 5 步新加的模块，5 分钟后原样提交，agent 本身的代码一行没动；记录里的工具、提示词版本、病例库大小都对。复核结论作为标签记在这次考试上，笔记只认带这个标签的 `-dirty` 记录

### 面试怎么说
> "The final exam was an ablation: the full agent once, then once without each new tool, on the same 200 cases, with the fixed pipeline re-run under the same, tightened masking rules."

---

## 第 2 步 · agent 是不是真的比固定流程强

### 在干什么
定稿 agent 和固定流程，同样 200 道题逐题比。

### 背景
第 15 章：agent 39.0%，固定流程 35.5%，逐题比 14 比 7，`p = 0.19`，不能排除碰巧。

### 执行

In [ ]:
fixed = runs["fixed-pipeline"]["run_id"]
for name in ("seeded-agent", "seeded-agent (without search_disease_texts)"):
    for k in (1, 10):
        run(f"uv run raredx eval compare {runs[name]['run_id']} {fixed} --k {k}")

### 结果怎么看

| 和固定流程逐题比 | 第 1 名 | 前 10 名 |
|---|---|---|
| 定稿 agent（全部工具） | 15 比 3，**p = 0.008** | 12 比 3，**p = 0.035** |
| 去掉"查疾病介绍"后（最终配置） | 12 比 2，**p = 0.013** | 10 比 2，**p = 0.039** |

**这是项目里第一次用数据证明"为什么需要 agent"**：同样的题、同样的防泄露规则，agent 比固定流程答对的显著更多，第 1 名和前 10 名都是。

### 面试怎么说
> "Against the fixed pipeline, on the same 200 cases and masking rules, the final agent wins 12 cases to 2 at first place (p = 0.013) and 10 to 2 in the top ten (p = 0.039). In chapter 15 the lead was 14 to 7, p = 0.19 — not established; now it is."

---

## 第 3 步 · 每个工具值多少分

### 在干什么
定稿 agent 和每个"拿掉一个工具"的版本逐题比。

### 执行

In [ ]:
full = runs["seeded-agent"]["run_id"]
for name in ("seeded-agent (without find_similar_patients)", "seeded-agent (without search_disease_texts)"):
    run(f"uv run raredx eval compare {full} {runs[name]['run_id']} --k 1")

### 结果怎么看

| 工具 | 拿掉以后第 1 名 | 逐题比（有它对 : 没它对） | 结论 |
|---|---|---|---|
| 找相似病人 | 41.5% → 38.5% | 10 : 4，p = 0.18 | **多半有用**，还不显著；保留 |
| 查疾病介绍 | 41.5% → 40.5% | 4 : 2，p = 0.69 | **基本没用**，还多花 18% 的 token；**从 agent 里去掉** |

- "查疾病介绍"的结果和第 18 章单独考检索时一致：文字介绍分不清亚型，找病的能力本来就弱
- 去掉以后，它在 MCP 服务里照常提供：人看介绍、引用出处时还是有用的
- 代码里改成：agent 默认不带它，要用时加 `--with-texts`

### 公司里什么情况怎么做
- **不加分就删**：每个工具都有成本（提示词变长、每轮多花 token、agent 多一个可能走偏的选项）
- **删也要留证据**：代码里写明"第 20 章测过，4 比 2，p = 0.69，多花 18% token"，以后的人知道为什么

### 面试怎么说
> "The ablation kept one tool and cut one: similar-patient retrieval was worth about three points at first place, while the text RAG was worth nothing measurable and cost 18% more tokens, so the agent no longer gets it by default."

---

## 第 4 步 · 错题分析：瓶颈挪到了哪

### 在干什么
和第 16 章一样，把答错的题按"错在哪一步"分类，和第一个 agent 比。

### 执行
大约一分钟：要读两次考试的全部记录，并对照资料库。

In [ ]:
import collections, json, re
from raredx.data import cases as case_store, sources
from raredx.evals.harness import candidates_of
from raredx.knowledge import hpo as hpo_knowledge

registry = sources.load_sources(ROOT / "evals/datasets.toml")
hpo = hpo_knowledge.load(sources.raw_dir(registry["hpo"], ROOT / "data"))
cases = {c.id: c for c in case_store.read(sources.processed_dir(registry["phenopacket-store"], ROOT / "data"))}

def buckets(run_id):
    counts = collections.Counter()
    for t in harness.traces(run_id):
        truth = next(a.value for a in t.info.assessments if a.name == "disease_id")
        case = cases[t.data.spans[0].inputs["question"]["case_id"]]
        answer = [c.disease_id for c in candidates_of(t.data.spans[0].outputs)]
        pattern = re.compile(re.escape(truth) + r"(?!\d)")
        tools = [s for s in t.data.spans if s.span_type == "TOOL"]
        seen = any(pattern.search(json.dumps(s.outputs, default=str)) for s in tools)
        examined = any(s.name in ("check_match", "lookup_disease") and pattern.search(json.dumps(s.inputs, default=str))
                       for s in tools)
        if truth in hpo.known_only_from(frozenset({case.pmid})):
            counts["① 注定答不对（病只在本篇论文里描述过）"] += 1
        elif answer[:1] == [truth]:
            counts["✅ 答对"] += 1
        elif not seen:
            counts["② 没找到"] += 1
        elif examined:
            counts["③ 看到了、细查了，没排第一"] += 1
        else:
            counts["④ 列表里有，没细查"] += 1
    return counts

first = next(r for r in harness.runs("dev-200") if r["name"] == "agent"
             and harness.client.get_run(r["run_id"]).data.params.get("prompt_version") == "agent-v3")
old, new = buckets(first["run_id"]), buckets(runs["seeded-agent"]["run_id"])
print(f"{'':<26}第一个 agent   定稿 agent")
for kind in sorted(set(old) | set(new)):
    print(f"{kind:<22} {old[kind]:>8} {new[kind]:>12}")

### 结果怎么看

| | 第一个 agent | 定稿 agent |
|---|---|---|
| ✅ 答对 | 78 | 83 |
| ① 注定答不对 | 40 | 40 |
| ② 没找到 | 40 | **23** |
| ③ 看到了、细查了，没排第一 | 29 | **46** |
| ④ 列表里有，没细查 | 13 | 8 |

- **新工具解决了"找"**：没找到的从 40 道降到 23 道；新开局让"没细查"从 13 降到 8
- **瓶颈挪到了"选"**：新找到的正确答案，大多数没有变成答对，而是变成了"看到了但没选对"（29 → 46）。这 46 道里，20 道正确答案排在第 2、3 名
- **满分是 80%**：40 道题的病只在本篇论文里描述过，按规则谁都答不对。按能答的 160 道算，定稿 agent 答对了一半多一点

翻这 46 道题的记录，大多是**两个很像的病选一个**（两种癫痫性脑病、IKAROS 家族的两种免疫缺陷），而且 agent 权衡证据的方式前后不一致：有时听"相似病人"的，有时听"逐条核对"的。这些病主要靠致病基因区分，只看症状本来就很难分开。

### 面试怎么说
> "The new tools fixed retrieval — cases whose right disease no tool ever showed fell from 40 to 23 — and moved the bottleneck to judgment: cases where the agent examined the right disease and ranked another first rose from 29 to 46, mostly close relatives that phenotype alone hardly separates."

---

## 第 5 步 · 重放判断：便宜地试三种判断方法

### 在干什么
瓶颈在"选"，那就试着改进判断。但不重跑整个 agent：从记录里把它查到的证据取出来，整理成一张证据表，只让模型**重新判断一次**。

### 背景
完整考一次约 ¥5（每道题最多 8 轮，每轮重发整段对话）。可我们要改的只是最后一步，证据已经在记录里了。只重放这一步，每道题调 1 次模型，便宜得多。

### 用了什么技术，为什么
- **证据表**（`evals/evidence.py`）：从记录里读出每个候选病的全部证据：按症状排第几、逐条核对的结果、几个相似病人得这个病（来自几篇论文）、介绍、基因、遗传方式。按证据排序，不按 agent 的偏好
- **三种判断方法**（`judge.py`）：
  1. 只给证据表，并写明怎么权衡，比如几个不同论文的相似病人都得这个病，是很强的证据
  2. 证据表 + agent 自己的排名，让它当复核医生，改掉不对的
  3. 第 1 种，加上思考模式
- 读的证据也记进追踪，照样过泄露审计

### 执行

In [ ]:
for name in ("judge-table", "judge-review", "judge-table-thinking"):
    record = latest(name)
    m = harness.client.get_run(record["run_id"]).data.metrics
    usd = ((m["usage/input_tokens"] - m["usage/cached_input_tokens"]) * 0.15 + m["usage/cached_input_tokens"] * 0.003
           + m["usage/output_tokens"] * 0.60) / 1e6
    print(f"{name:<22} 第 1 名 {m['disease_top1/mean']:.1%}   输出 {m['usage/output_tokens'] / 1e6:.2f}M token   "
          f"约 ${usd:.2f}（非高峰价）")
    run(f"uv run raredx eval compare {record['run_id']} {full} --k 1")

### 结果怎么看

| 判断方法 | 第 1 名 | 和 agent 自己的判断逐题比 | 花费 |
|---|---|---|---|
| agent 自己（对照） | 41.5% | | |
| ① 只给证据表 | 37.5% | 5 : 13，**更差** | 约 ¥1 |
| ② 证据表 + 复核 agent 的排名 | 40.5% | 2 : 4，基本照抄 | 约 ¥1 |
| ③ 证据表 + 思考模式 | 39.5% | 6 : 10，没变好 | **约 ¥19** |

**三种都没有赢**。agent 是边查边想的，它对证据的理解比一张压缩过的表多；把证据抽出来重新判断，信息反而丢了。剩下的"二选一"，只看症状本来就难分开。

这是一个**有价值的负面结果**：只花了约三次实验的钱，就排除了一条看起来很合理的路，不用把它做进 agent、再跑完整考试才发现没用。

### 公司里什么情况怎么做
- **只重放要改的那一步**：大部分改动只动了系统的一小块，用记录下来的输入单独测这一块，便宜得多
- **负面结果也要记下来**：下一个人不用再试一遍

### 面试怎么说
> "To try new ways of judging without re-running the agent, I replayed its recorded evidence through a one-call judge: evidence table, review of the agent's ranking, and thinking mode. None beat the agent's own judgment — a negative result bought for a fraction of a full run."

---

## 第 6 步 · 基因考法

### 在干什么
给定稿 agent 50 个候选基因（第 19 章的考法），和不用模型的基因排名比。

### 执行

In [ ]:
agent_g, rank_g = latest("seeded-agent", "dev-200+genes"), latest("gene-rank", "dev-200+genes")
for label, record in (("不用模型：按基因排名", rank_g), ("定稿 agent", agent_g)):
    m = harness.client.get_run(record["run_id"]).data.metrics
    print(f"{label:<12} 第 1 名 {m['disease_top1/mean']:.1%}   前 10 名 {m['disease_top10/mean']:.1%}")
for k in (1, 10):
    run(f"uv run raredx eval compare {agent_g['run_id']} {rank_g['run_id']} --k {k}")

### 结果怎么看

| 基因考法 | 第 1 名 | 前 10 名 |
|---|---|---|
| 不用模型：按基因排名 | 62.0% | 78.5% |
| 定稿 agent | 62.5% | 75.5% |

- 第 1 名 14 比 13，打平；前 10 名 2 比 8，agent 略差（p = 0.11）
- **有了基因，最难的"二选一"已经被基因排名解决了**，agent 能加的空间很小。前 10 名略差，是因为 agent 列不满 10 个时，用的是表型工具的顺序补，而不是基因排名的顺序
- 这也回答了第 4 步：只看症状分不开的病，医院靠基因检测来分
- 后续见第 8 步：agent 和基因排名答对的题不一样，把两个排名合并，比任何一个单独用都好

### 面试怎么说
> "With fifty candidate genes the agent ties a no-model gene ranking at first place: sequencing already resolves the close calls that phenotype cannot. Its slightly weaker top ten comes from filling short lists in phenotype order rather than gene order — a fix for later."

---

## 第 7 步 · 两次意外

### 意外一：考到一半，余额用完了
- **发生了什么**：DeepSeek 账户余额耗尽，接口开始拒绝调用（`402 Insufficient Balance`）。一次考试有 16 道题因此没答出来，被当成"答错"；固定流程那次 200 道全被拒，记成了 0 分
- **问题在哪**：接口故障被当成了系统的错误，分数就不是系统的真实水平
- **修法**：考试程序把"因为接口故障没答"的题单独数出来（`errors/model_api`），只要有一道，这次考试就判作废，并提示原因。系统自己的错误（比如没交答案）照样算答错。无效的记录全部删掉，充值后重考

### 意外二：一次实验超支
- **发生了什么**：三种判断方法我预计一共约 ¥3，实际约 ¥22。思考模式那一种，模型每道题"思考"了两万多个 token，输出又是最贵的部分，一种就花了约 ¥19
- **问题在哪**：按普通回答的长度估了价；花费上限是在两次考试之间检查的，挡不住一次考试内部超支
- **规矩**：换了新的用法（思考模式、新模型、新的提示词形状），**先在 ci-20 上试，看清每道题的单价，再上 dev-200**

### 执行

In [ ]:
for name in ("seeded-agent", "fixed-pipeline", "judge-table-thinking"):
    run_data = harness.client.get_run(latest(name)["run_id"]).data
    print(f"{name:<22} 接口故障 {run_data.metrics.get('errors/model_api', 0):.0f} 道   泄露审计 {run_data.tags.get('leak_audit')}")

### 结果怎么看
- 留下来的考试，接口故障都是 0、泄露审计都通过
- 以后每次考试的命令行，最后都会报这两项

### 公司里什么情况怎么做
- **分数要分清"系统错了"和"环境坏了"**：限流、超时、欠费都不是模型的问题，混在一起，分数就失真了
- **新的用法先小规模试跑看成本**：大模型的花费主要看输出长度，思考模式会让输出多出几十倍

### 面试怎么说
> "Two incidents made the harness stricter: when the API account ran dry mid-exam, refused calls had been scored as misses, so model-API failures are now counted apart and void the run; and after a thinking-mode run cost six times its estimate, any new cost profile is piloted on 20 cases first."

---

## 第 8 步 · 定稿前：还有没有能提高的地方

### 在干什么
留出集只能考一次，考完系统就不能再改。所以开考之前，把还能想到的改进方向用已有的记录都试一遍，不花钱。

### 背景
前面几步的考试记录里，存着每道题每个系统的完整答案。很多"改进"其实只是把已有的答案换一种方式组合，或者多给一点病例里本来就有的信息。这些都可以直接在记录上算，不用再调模型。

### 用了什么技术，为什么
试了五个方向：

| 方向 | 想法 |
|---|---|
| 合并排名（只看症状） | agent 的排名和不用模型的排名各有对错，合起来会不会更好 |
| 换一种方式补满 10 个 | 补的时候把"找相似病人"的结果也用上 |
| 合并排名（基因考法） | agent 和基因排名答对的题不一样（第 6 步：14 比 13） |
| 加上"明确没有的症状" | 病例里记了病人**没有**哪些症状，我们一直没用 |
| 加上发病年龄 | 提示词要求判断"发病年龄是否符合"，可 agent 并不知道病人几岁发病 |

合并用的是**倒数排名融合**（RRF）：一个病在两份名单里都靠前，就排最前。常数用通用的 60，不调参数。

### 公司里什么情况怎么做
- **正式评测之前做一轮"还能不能改"的排查**：考完就不能改了，排查要在之前做完
- **排查尽量用已有的记录**：不花钱，一个下午能试很多方向
- **不调参数**：在 200 道题上调出来的参数，换一批题多半不灵

### 执行
先看合并排名。用的就是系统里现在的合并函数 `fuse`。

In [ ]:
from raredx.agent import fuse

def recorded(record):
    out = {}
    for t in harness.traces(record["run_id"]):
        truth = next(a.value for a in t.info.assessments if a.name == "disease_id")
        out[t.data.spans[0].inputs["question"]["case_id"]] = (truth, candidates_of(t.data.spans[0].outputs))
    return out

def rate(mine, other=None):
    top1 = top10 = 0
    for cid, (truth, answer) in mine.items():
        ids = [c.disease_id for c in (fuse(answer, other[cid][1]) if other else answer)]
        top1 += ids[:1] == [truth]
        top10 += truth in ids[:10]
    return f"第 1 名 {top1 / len(mine):.1%}   前 10 名 {top10 / len(mine):.1%}"

final = recorded(runs["seeded-agent (without search_disease_texts)"])
print("只看症状")
print("  最终 agent                 ", rate(final))
print("  和「找相似病人」的排名合并   ", rate(final, recorded(latest("similar-patients"))))
print("  和「先排名再核对」的排名合并 ", rate(final, recorded(latest("rank-then-check"))))
with_genes, gene_ranking = recorded(agent_g), recorded(rank_g)
print("症状 + 基因")
print("  agent（第 6 步那次考试）     ", rate(with_genes))
print("  和基因排名合并              ", rate(with_genes, gene_ranking))
print("  agent 列的答案不满 10 个的题：", sum(len(answer) < 10 for _, answer in final.values()), "道")

### 结果怎么看

| | 第 1 名 | 前 10 名 |
|---|---|---|
| **只看症状**：最终 agent | 40.5% | 58.5% |
| 和"找相似病人"合并 | 34.5% | 60.5% |
| 和"先排名再核对"合并 | 35.0% | 58.5% |
| **症状 + 基因**：agent | 62.5% | 75.5% |
| **和基因排名合并** | **67.0%** | **78.0%** |

- **只看症状，合并反而变差**：不用模型的排名比 agent 弱得多（第 1 名约 30% 对 40%），合进来是拖后腿
- **"补满 10 个"没有空间了**：现在每道题的答案都已经是满的 10 个
- **基因考法里，合并是真的改进**：基因排名和 agent 差不多强，答对的题又不一样。逐题比，合并后对 agent 单独是 13 比 4（p = 0.049），对基因排名单独是 11 比 1（p = 0.006）

所以这一条做进了系统：题目里有候选基因时，agent 的最终答案会和基因排名合并。这也改写了第 6 步的结论：有基因时 agent 不是没用，而是要和基因排名一起用。

再看"明确没有的症状"。

In [ ]:
import zipfile

archive = next((ROOT / "data/raw/phenopacket-store").rglob("all_phenopackets.zip"))
absent = {}
with zipfile.ZipFile(archive) as z:
    for name in z.namelist():
        if name.endswith(".json"):
            packet = json.loads(z.read(name))
            if packet.get("id") in final:
                absent[packet["id"]] = [f["type"]["id"] for f in packet.get("phenotypicFeatures", []) if f.get("excluded")]
print(f"200 道题里，{sum(bool(v) for v in absent.values())} 道记录了「明确没有的症状」，一共 {sum(map(len, absent.values()))} 个\n")

def clashes(disease, case):
    # 这个病一半以上的病人都有、而这个病人明确没有的症状，有几个
    typical = {f.term for f in hpo.features(disease, frozenset({case.pmid}))
               if not f.absent and (f.frequency.fraction or 0) >= 0.5}
    return sum(any(hpo.relation(t, a) in ("same", "more specific") for t in typical) for a in absent.get(case.id, []))

against_wrong = against_right = no_difference = 0
for cid, (truth, answer) in final.items():
    ids = [c.disease_id for c in answer]
    if truth in ids[1:10] and ids[0] in hpo.names:      # 正确答案在名单里，但不是第 1 名
        wrong, right = clashes(ids[0], cases[cid]), clashes(truth, cases[cid])
        against_wrong += wrong > right
        against_right += right > wrong
        no_difference += wrong == right
print(f"正确答案排在第 2–10 名的题，一共 {against_wrong + against_right + no_difference} 道。「没有的症状」：")
print(f"  更不利于 agent 选错的那个病：{against_wrong} 道")
print(f"  更不利于正确答案：          {against_right} 道")
print(f"  没有区别：                  {no_difference} 道")

### 结果怎么看
- 这类信息很多：200 道题里 163 道有，一共 1,532 个
- **可是它更不利于正确答案**（14 道），而不是 agent 选错的那个病（9 道）

为什么会这样？论文作者记录"这个病人没有癫痫"，往往因为癫痫正是这个病的典型症状。整理病例的人是照着**这个病的症状表**逐项打勾打叉的。所以"没有的症状"这份清单，本身就在暗示答案是哪个病：一个模型看到"没有 A、没有 B、没有 C"，就能猜出这是哪个病的检查表。

这是一种很隐蔽的泄露。**第 05 章读病例时把这类症状丢掉，是对的，不能加回来。**

同样的办法也试了**发病年龄**（162 道题有记录）：在 29 道"差一点"的题里，年龄支持正确答案的只有 2 道，支持 agent 选错的病也是 2 道。很像的病，发病年龄也差不多，帮不上。

### 小结：定稿

| 方向 | 结果 |
|---|---|
| 合并排名（只看症状） | 变差，不用 |
| 换一种方式补满 10 个 | 没有空间 |
| **合并排名（基因考法）** | **第 1 名 62.5% → 67.0%，做进系统** |
| 加上"明确没有的症状" | 不利于正确答案，而且会泄露，不加 |
| 加上发病年龄 | 帮不上，不加 |

只看症状的部分，便宜的办法都试完了，就此定稿。还没试的只有换更贵的模型（贵 4–5 倍），决定不试：就算它好几个百分点，以后每次考试、包括留出集，都要贵 4–5 倍。

### 面试怎么说
> "Before freezing the system for the held-out run, I replayed five more ideas on the recorded answers at no cost. One was a real gain: with sequencing data, fusing the agent's ranking with the gene ranking by reciprocal rank took top-1 from 62.5% to 67.0% (13 cases to 4, p = 0.049). One was a trap: the patients' recorded-absent findings count against the right disease more often than the wrong one, because curators tick them off the true disease's own feature table — a leak, so they stay out."

---

## 本章小结

| 问题 | 答案 |
|---|---|
| agent 有没有比固定流程强？ | **有**：12 比 2（p = 0.013），前 10 名 10 比 2（p = 0.039） |
| 每个工具值多少？ | 找相似病人约 +3 个百分点，保留；查疾病介绍不加分，从 agent 里去掉 |
| 剩下的错在哪？ | 从"没找到"挪到了"没选对"（29 → 46），多是很像的两个病 |
| 判断能改进吗？ | 试了三种，都没赢：负面结果，记录在案 |
| 有基因时呢？ | agent 单独 62.5%，和基因排名打平；两个排名合并以后 **67.0%**，显著好于任何一个 |
| 定稿前还能提高吗？ | 只看症状：试了 5 个方向，都没用，就此定稿。基因考法：合并排名，+4.5 个百分点 |
| 学到了什么？ | 接口故障要单独数；新的用法先小考看成本；"明确没有的症状"会泄露答案 |

最终配置：新开局 + 4 个核心工具 + 找相似病人；基因考法再加基因排名工具，并把 agent 的排名和基因排名合并。dev-200 上：只看症状，第 1 名 40.5%、前 10 名 58.5%；有基因，第 1 名 67.0%、前 10 名 78.0%（按记录重放）。

## 面试题

1. **Can you prove the agent is better than a pipeline?**
   > "Yes, now: on 200 cases with the same masking, 12 to 2 at first place, p = 0.013. Earlier it was 14 to 7, p = 0.19, and I said so."

2. **How did you decide which tools to keep?**
   > "Ablation: remove one, rerun, compare case by case. Similar-patient retrieval stayed; text RAG went — no measurable gain, 18% more tokens."

3. **What limits the agent now?**
   > "Judgment between close relatives that phenotype alone barely separates — and twenty percent of cases no tool can answer under strict masking. With sequencing data the close calls mostly resolve."

4. **How do you keep evaluation costs down while iterating?**
   > "Replay: re-run only the step being changed on recorded inputs; smoke-test on 20 cases; pilot any new cost profile before the full set; full runs only for finalists."

**下一章 21 · 做成服务**：把 agent 和工具做成一个能演示的服务：一个网页，输入症状，实时看 agent 每一步查了什么、为什么这么判断，最后给出带出处的诊断。换一台电脑，一条命令就能跑起来。